<a href="https://colab.research.google.com/github/harskarodlan/DeepLearning/blob/assignment_2/Assignment_2/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [16]:
%cd "/content/drive/MyDrive/Colab Notebooks/DL/Assignment_2"

/content/drive/MyDrive/Colab Notebooks/DL/Assignment_2


In [17]:
import numpy as np
import pickle
import copy
import matplotlib.pyplot as plt


In [18]:
import os
print(os.getcwd())

/content/drive/MyDrive/Colab Notebooks/DL/Assignment_2


In [19]:
from torch_gradient_computations import ComputeGradsWithTorch

In [20]:
from ann import *
from data_handling import *
from plotting import *
from ann_sigmoid import *

# Basic

 ## Load data

In [21]:

cifar_dir = '../Datasets/cifar-10-batches-py/'


For 1 batch:

In [ ]:
# @title
trainX, trainY, trainy = LoadBatch(cifar_dir +  'data_batch_1')
validX, validY, validy = LoadBatch(cifar_dir +  'data_batch_2')

For all batches:

In [22]:
# improvement 2.1a: use all training batches
X, Y, y = LoadAll(cifar_dir)
trainX = X[:, 1000:]
trainY = Y[:, 1000:]
trainy = y[1000:]
validX = X[:, :1000]
validY = Y[:, :1000]
validy = y[:1000]


In [14]:
testX, testY, testy = LoadBatch(cifar_dir +  'test_batch')


d = trainX.shape[0]
n = trainX.shape[1]
K = trainY.shape[0]

## Normalize data

In [23]:
mean_X = np.mean(trainX, axis=1).reshape(d, 1)
std_X = np.std(trainX, axis=1).reshape(d, 1)

trainX = NormalizeData(trainX, mean_X, std_X)
validX = NormalizeData(validX, mean_X, std_X)
testX = NormalizeData(testX, mean_X, std_X)

## Initialize parameters

In [25]:
# creat random generator object
rng = np.random.default_rng()

# get the BitGenerator used by default_rng
BitGen = type(rng.bit_generator)
# use a seed and save it
# makes initilization repeatable
seed = 42
# use the state from a fresh bit generator
rng.bit_generator.state = BitGen(seed).state

# network represented by dict to hold parameters: keys 'W', 'b'
init_net = {}
# W is (K, d): one weight per class and input feature
# Initialize W randomly normally distributed
init_net['W'] = .01*rng.standard_normal(size = (K, d))  # (K, d)
init_net['W'] = init_net['W'].astype(np.float32)
# b is (K, 1): one bias per class
# initialize b to zero
init_net['b'] = np.zeros((K, 1), dtype=np.float32)                        # (K, 1)


## Gradient Descent

In [ ]:

lambdas = [0, 0, .1, 1]
etas = [.1, .001, .001, .001]

for i in range(4):

    # ----- 8: Mini batch gradient descent -----

    GDparams = {'n_batch': 100, 'eta': etas[i], 'n_epochs': 40}

    lam = lambdas[i]

    # train network
    trained_net, history = MiniBatchGD(trainX, trainY, trainy,
                                    validX, validY, validy,
                                    GDparams, init_net, lam, seed=42)

    # test network
    P_test = ApplyNetwork(testX, trained_net)
    test_acc = ComputeAccuracy(P_test, testy)
    print(f"test accuracy: {100 * test_acc:.2f}%")


    # ------ Plotting statistics --------

    epochs = np.arange(1, GDparams['n_epochs'] + 1)

    plt.figure()
    plt.plot(epochs, history['train_loss'], label='training loss')
    plt.plot(epochs, history['val_loss'], label='validation loss')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.title('Training and validation loss')
    plt.legend()
    plt.savefig(f'loss_{i}')
    #plt.show()

    plt.figure()
    plt.plot(epochs, history['train_cost'], label='training cost')
    plt.plot(epochs, history['val_cost'], label='validation cost')
    plt.xlabel('epoch')
    plt.ylabel('cost')
    plt.title('Training and validation cost')
    plt.legend()
    plt.savefig(f'cost_{i}')
    #plt.show()

    VisualizeWeights(trained_net, f'W_visualized_{i}')

# BONUS POINTS:

## 2.1c: Grid Search

In [ ]:

# improvement 2.1c: grid search
lambda_grid = [0, 1e-4, 1e-3, 1e-2]
eta_grid = [5e-4, 1e-3]
batch_grid = [50, 100]

results = []

for lam in lambda_grid:
    for eta in eta_grid:
        for n_batch in batch_grid:
            # ----- 8: Mini batch gradient descent -----

            print("----------------------------------------")
            print(f"testing parameters: lam={lam:.6f}, eta={eta:.6f}, n_batch={n_batch}")
            print("----------------------------------------")

            GDparams = {'n_batch': n_batch, 'eta': eta, 'n_epochs': 40}

            # train network
            trained_net, history = MiniBatchGD(trainX, trainY, trainy,
                                            validX, validY, validy,
                                            GDparams, init_net, lam, seed=42, flip=True)

            # training accuracy
            P_train = ApplyNetwork(trainX, trained_net)
            train_acc = ComputeAccuracy(P_train, trainy)

            # validation accuracy
            P_val = ApplyNetwork(validX, trained_net)
            val_acc = ComputeAccuracy(P_val, validy)

            results.append({
                'lam': lam,
                'eta': eta,
                'n_batch': n_batch,
                'val_acc': val_acc,
                'net': trained_net
            })

            print(f"lam={lam}, eta={eta}, n_batch={n_batch},",
                   f"val_acc={100*val_acc:.2f}, train_acc={100*train_acc:.2f}")


best_result = max(results, key=lambda r: r['val_acc'])
best_net = best_result['net']

P_test = ApplyNetwork(testX, best_net)
test_acc = ComputeAccuracy(P_test, testy)


# training accuracy
P_train = ApplyNetwork(trainX, best_net)
train_acc = ComputeAccuracy(P_train, trainy)
print(f"Training accuracy: {100 * train_acc:.2f}%")

# validation accuracy
P_val = ApplyNetwork(validX, best_net)
val_acc = ComputeAccuracy(P_val, validy)
print(f"Validation accuracy: {100 * val_acc:.2f}%")

print("Best parameters:")
print(best_result['lam'], best_result['eta'], best_result['n_batch'])
print(f"Test accuracy: {100 * test_acc:.2f}%")

## 2.1b: Flip Images

Flipped:

In [ ]:

GDparams = {'n_batch': 100, 'eta': 0.001, 'n_epochs': 20}

# train network FLIPPED
trained_net, history = MiniBatchGD(trainX, trainY, trainy,
                                validX, validY, validy,
                                GDparams, init_net, 0.01, seed=42, flip=True)

print("FLIPPED: ")

# training accuracy
P_train = ApplyNetwork(trainX, trained_net)
train_acc = ComputeAccuracy(P_train, trainy)
print(f"Training accuracy: {100 * train_acc:.2f}%")

# validation accuracy
P_val = ApplyNetwork(validX, trained_net)
val_acc = ComputeAccuracy(P_val, validy)
print(f"Validation accuracy: {100 * val_acc:.2f}%")

P_test = ApplyNetwork(testX, trained_net)
test_acc = ComputeAccuracy(P_test, testy)
print(f"Test accuracy: {100 * test_acc:.2f}%")

Not flipped:

In [ ]:
GDparams = {'n_batch': 100, 'eta': 0.001, 'n_epochs':40}

# train network NOT FLIPPED
trained_net, history = MiniBatchGD(trainX, trainY, trainy,
                                validX, validY, validy,
                                GDparams, init_net, 0.01, seed=42, flip=False)

print("NOT FLIPPED: ")

# training accuracy
P_train = ApplyNetwork(trainX, trained_net)
train_acc = ComputeAccuracy(P_train, trainy)
print(f"Training accuracy: {100 * train_acc:.2f}%")

# validation accuracy
P_val = ApplyNetwork(validX, trained_net)
val_acc = ComputeAccuracy(P_val, validy)
print(f"Validation accuracy: {100 * val_acc:.2f}%")

P_test = ApplyNetwork(testX, trained_net)
test_acc = ComputeAccuracy(P_test, testy)
print(f"Test accuracy: {100 * test_acc:.2f}%")

## 2.2: Sigmoid + MBCE

Softmax:

In [ ]:
# Softmax
GDparams_sm = {'n_batch': 100, 'eta': 0.001, 'n_epochs': 40}

# train network FLIPPED
trained_net_sm, history_sm = MiniBatchGD(trainX, trainY, trainy,
                                validX, validY, validy,
                                GDparams_sm, init_net, 0.01, seed=42, flip=True)



P_test_sm = ApplyNetwork(testX, trained_net_sm)
test_acc_sm = ComputeAccuracy(P_test_sm, testy)
print(f"Test accuracy: {100 * test_acc_sm:.2f}%")


Sigmoid:

In [ ]:
# Sigmoid
GDparams_sig = {'n_batch': 100, 'eta': 0.01, 'n_epochs': 40}

# train network FLIPPED
trained_net_sig, history_sig = MiniBatchGDSigmoid(trainX, trainY, trainy,
                                validX, validY, validy,
                                GDparams_sig, init_net, 0.01, seed=42, flip=True)



P_test_sig = ApplyNetworkSigmoid(testX, trained_net_sig)
test_acc_sig = ComputeAccuracy(P_test_sig, testy)
print(f"Test accuracy: {100 * test_acc_sig:.2f}%")

### Histograms:

In [ ]:
gt_probs_sm = P_test_sm[testy, np.arange(P_test_sm.shape[1])]
preds_sm = np.argmax(P_test_sm, axis=0)

correct_probs_sm = gt_probs_sm[preds_sm == testy]
incorrect_probs_sm = gt_probs_sm[preds_sm != testy]


gt_probs_sig = P_test_sig[testy, np.arange(P_test_sig.shape[1])]
preds_sig = np.argmax(P_test_sig, axis=0)

correct_probs_sig = gt_probs_sig[preds_sig == testy]
incorrect_probs_sig = gt_probs_sig[preds_sig != testy]


plt.figure()
plt.hist(correct_probs_sm, bins=30, alpha=0.7, label='correct')
plt.hist(incorrect_probs_sm, bins=30, alpha=0.7, label='incorrect')
plt.xlabel('probability of ground-truth class')
plt.ylabel('count')
plt.title('Softmax: ground-truth class probabilities on test set')
plt.legend()
plt.savefig('softmax_hist.png')
plt.show()

plt.figure()
plt.hist(correct_probs_sig, bins=30, alpha=0.7, label='correct')
plt.hist(incorrect_probs_sig, bins=30, alpha=0.7, label='incorrect')
plt.xlabel('probability of ground-truth class')
plt.ylabel('count')
plt.title('Sigmoid/MBCE: ground-truth class probabilities on test set')
plt.legend()
plt.savefig('sigmoid_hist.png')
plt.show()

### Loss Plots

In [ ]:
epochs = np.arange(1, GDparams_sig['n_epochs'] + 1)

plt.figure()
plt.plot(epochs, history_sm['train_loss'], label='softmax train')
plt.plot(epochs, history_sm['val_loss'], label='softmax val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training and validation loss')
plt.legend()
plt.savefig('softmax_loss.png')
plt.show()


plt.figure()
plt.plot(epochs, history_sig['train_loss'], label='sigmoid train')
plt.plot(epochs, history_sig['val_loss'], label='sigmoid val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training and validation loss')
plt.legend()
plt.savefig('sigmoid_loss.png')
plt.show()